In [14]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd

from utils.text_preprocessing import clean_text

In [15]:
model = joblib.load(
    "../trained_models/fake_job_detector.pkl"
)

vectorizer = joblib.load(
    "../trained_models/tfidf_vectorizer.pkl"
)

In [16]:
type(model)

sklearn.calibration.CalibratedClassifierCV

In [17]:
type(vectorizer)

sklearn.feature_extraction.text.TfidfVectorizer

In [19]:
sample_job = """
Google is hiring a Software Engineer.

Requirements:
Python
Machine Learning
TensorFlow

Benefits:
Health insurance
Remote work
Paid leave
"""

In [20]:
cleaned_job = clean_text(sample_job)

print(cleaned_job)

google is hiring a software engineer requirements python machine learning tensorflow benefits health insurance remote work paid leave


In [21]:
job_vector = vectorizer.transform([cleaned_job])

In [22]:
prediction = model.predict(job_vector)

prediction

array([0])

In [23]:
def predict_job(job_description):
    """
    Predict whether a job posting is Genuine or Fraudulent.
    """

    # Clean text
    cleaned_text = clean_text(job_description)

    # TF-IDF transformation
    text_vector = vectorizer.transform([cleaned_text])

    # Prediction
    prediction = model.predict(text_vector)[0]

    # Prediction probability
    probabilities = model.predict_proba(text_vector)[0]

    # Confidence score
    confidence = probabilities[prediction]

    # Label
    label = "Fraudulent Job" if prediction == 1 else "Genuine Job"

    return label, confidence, probabilities

In [24]:
label, confidence, probabilities = predict_job(sample_job)

print("Prediction :", label)
print(f"Confidence : {confidence:.2%}")
print("Probabilities :", probabilities)

Prediction : Genuine Job
Confidence : 98.83%
Probabilities : [0.98827683 0.01172317]


In [25]:
def get_risk_level(probability):
    """
    Returns risk level based on fraud probability.
    """

    if probability < 0.30:
        return "🟢 Low Risk"

    elif probability < 0.70:
        return "🟡 Medium Risk"

    else:
        return "🔴 High Risk"

In [26]:
label, confidence, probabilities = predict_job(sample_job)

fraud_probability = probabilities[1]

risk = get_risk_level(fraud_probability)

print("=" * 50)
print("Fake Job Detector")
print("=" * 50)

print("Prediction :", label)
print(f"Confidence : {confidence:.2%}")
print(f"Fraud Probability : {fraud_probability:.2%}")
print("Risk Level :", risk)

Fake Job Detector
Prediction : Genuine Job
Confidence : 98.83%
Fraud Probability : 1.17%
Risk Level : 🟢 Low Risk
